In [ ]:
import cv2
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import pandas as pd

def calculate_blur(img_path):
    img_path_str = str(img_path)

    try:
        img = cv2.imread(img_path_str, cv2.IMREAD_GRAYSCALE)

        if img is None:
            return img_path_str, -1.0, "Read Error"

        focus_score = cv2.Laplacian(img, cv2.CV_64F).var()

        return img_path_str, focus_score, "OK"
    
    except Exception as e:
        return img_path_str, -1.0, str(e)
    
def run_focus_scoring(file_list):
    results = []
    max_workers = os.cpu_count() or 4
    print(f"using {max_workers} cpus to compute focus score")

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(calculate_blur, path): path for path in file_list}

        for future in tqdm(as_completed(futures), total=len(futures), desc="calculating focus score"):
            path_str, score, status = future.result()
            results.append({
                "file_path":path_str,
                "focus_score":score,
                "status":status
            })
    
    df = pd.DataFrame(results)
    df.to_csv("./focus_scores.csv", index=False)
    print(f"complete calculating focus score")

    return df 